In [16]:
# conda activate meteor
# rm -r venv              
# make first-venv
# make clean
# make virtual-environment
# source venv/bin/activate
# then select the kernel 'venv' in jupyter notebook

import os
import sys
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
import pandas as pd
from dataclasses import asdict
import warnings
from pandas.errors import SettingWithCopyWarning
from functools import partial
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=(SettingWithCopyWarning))
warnings.filterwarnings("ignore", message=".*not in pamset.*")

from meteor import MeteorPatternScaling, meteor_plot_utils
from meteor import prpatt
from meteor import Cmip6MeteorDataGetter
from meteor import scm_forcer_engine
from ciceroscm import CICEROSCM
from ciceroscm import input_handler

from meteor.prpatt import global_mean
from meteor.prpatt import recon

import cartopy.crs as ccrs
import cartopy as cart
from cartopy.util import add_cyclic_point
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
import seaborn as sns
from matplotlib.gridspec import GridSpec
from scipy.interpolate import griddata
import matplotlib.ticker as mticker
from scipy.stats import pearsonr
from tqdm import tqdm

In [17]:
cachedir = os.path.join(os.getcwd(), "cache")
if not os.path.exists(cachedir):
        os.makedirs(cachedir)
        
figdir = os.path.join(os.getcwd(), "figures")
if not os.path.exists(figdir):
        os.makedirs(figdir)

cscm_data_dir = os.path.join(os.getcwd(), "..", "src", "meteor", "default_scm_data")

# Data preparations

### Getting some data that can be used by the forcer engine and ciceroscm

SSP245 for aerorol residual from abrupt-4xCO2

In [18]:
%%capture
ih = input_handler.InputHandler({})
# NBVAL_IGNORE_OUTPUT

#For predictions: 
conc_data = input_handler.read_inputfile(os.path.join(cscm_data_dir, "ssp245_conc_RCMIP.txt"))
em_data = ih.read_emissions(os.path.join(cscm_data_dir, "ssp245_em_RCMIP.txt"))

standard SSP scenarios

In [19]:
scenarios = [ "ssp126", "ssp245", "ssp370", "ssp585"]

In [20]:
%%capture
#For predictions: read in SSP concentrations and emissions
#scenarios = ["ssp126","ssp245","ssp370","ssp585"]

ih = input_handler.InputHandler({})
# NBVAL_IGNORE_OUTPUT

conc_data_fwd=[]
em_data_fwd=[]
for i,s in enumerate(scenarios):
    conc_data_fwd.append(input_handler.read_inputfile(os.path.join(cscm_data_dir, s+"_conc_RCMIP.txt")))
    em_data_fwd.append(ih.read_emissions(os.path.join(cscm_data_dir, s+"_em_RCMIP.txt")))

SSP534-over

In [21]:
%%capture
#For predictions: read in SSP concentrations and emissions

ih = input_handler.InputHandler({})
# NBVAL_IGNORE_OUTPUT

conc_data_ssp534=[]
em_data_ssp534=[]
for i,s in enumerate(['ssp534-over']):
    conc_data_ssp534.append(input_handler.read_inputfile(os.path.join(cscm_data_dir, s+"_conc_RCMIP.txt")))
    em_data_ssp534.append(ih.read_emissions(os.path.join(cscm_data_dir, s+"_em_RCMIP.txt")))

### Getting data for standard scenarios

Getting CMIP6 data from the zarrstore for tas

In [22]:
flds = ["tas","pr"]

In [23]:
data_getter = Cmip6MeteorDataGetter(exps=["piControl", "abrupt-4xCO2", "historical", "ssp126", "ssp245", "ssp370", "ssp585"], 
                                    flds = flds, 
                                    dbe=['CMIP','CMIP','CMIP', 'ScenarioMIP', 'ScenarioMIP','ScenarioMIP', 'ScenarioMIP'])

#for property, value in vars(data_getter).items():
#    print(property)

In [24]:
models = data_getter.models
len(models)

35

Create training data dictionary for all selected models (may run for a while) - and save locally in 'cache' (cachedir) due to data memory requirements

In [25]:
# remove problematic models when creating the pattern
if 'IITM-ESM' in models: models.pop(models.index('IITM-ESM'))  # drop because bad data
if 'IPSL-CM6A-LR' in models: models.pop(models.index('IPSL-CM6A-LR'))  # drop because of data issue
if 'TaiESM1' in models: models.pop(models.index('TaiESM1'))  # drop because of data issue
if 'EC-Earth3' in models: models.pop(models.index('EC-Earth3'))  # drop because of data issue in pr
if 'GISS-E2-1-G' in models: models.pop(models.index('GISS-E2-1-G'))  # drop temperature fit doesn't work
if 'MIROC6' in models: models.pop(models.index('MIROC6'))
if 'MPI-ESM1-2-LR' in models: models.pop(models.index('MPI-ESM1-2-LR'))
if 'FGOALS-g3' in models: models.pop(models.index('FGOALS-g3'))    # data offset between piC and hist
if 'MRI-ESM2-0' in models: models.pop(models.index('MRI-ESM2-0'))

In [26]:
training_data_dict={}
for nm,mname in enumerate(models):
    #print(nm,mname)

    training_data_dict[mname] = {
        "base": xr.open_dataset(os.path.join(cachedir, f"{mname}_base_training_data.nc")),
        "co2x4": xr.open_dataset(os.path.join(cachedir, f"{mname}_co2x4_training_data.nc")),
        "sulxanom": xr.open_dataset(os.path.join(cachedir, f"{mname}_sulxanom_training_data.nc")),
        "ssp126": xr.open_dataset(os.path.join(cachedir, f"{mname}_ssp126_training_data.nc")),
        "ssp245": xr.open_dataset(os.path.join(cachedir, f"{mname}_ssp245_training_data.nc")),
        "ssp370": xr.open_dataset(os.path.join(cachedir, f"{mname}_ssp370_training_data.nc")),
        "ssp585": xr.open_dataset(os.path.join(cachedir, f"{mname}_ssp585_training_data.nc")),
    }

# Evaluation of timescale separation

In [ ]:
%%capture

timescales_ghg=[1,2,3,4]
timescales_aer=[0,1,2,3,4]
tser_og_ghg = np.zeros([len(flds),len(models),len(timescales_ghg),100]) * np.nan
tser_fit_ghg = np.zeros([len(flds),len(models),len(timescales_ghg),100]) * np.nan
mae_ghg = np.zeros([len(flds),len(models),len(timescales_ghg)]) * np.nan
rmse_ghg = np.zeros([len(flds),len(models),len(timescales_ghg)]) * np.nan
tser_og_aer = np.zeros([len(flds),len(models),len(timescales_ghg),len(timescales_aer),250]) * np.nan
tser_fit_aer = np.zeros([len(flds),len(models),len(timescales_ghg),len(timescales_aer),250]) * np.nan
mae_aer = np.zeros([len(flds),len(models),len(timescales_ghg),len(timescales_aer)]) * np.nan
rmse_aer = np.zeros([len(flds),len(models),len(timescales_ghg),len(timescales_aer)]) * np.nan
CMIP_pattern_dict_tscales = {}

for nm,mname in enumerate(models):

    # get original timeseries output first for error metrics calculations ()
    tser_og_ghg_co2x4 = np.zeros([len(flds),100])
    tser_og_aer_sulxanom = np.zeros([len(flds),250])

    tmp = MeteorPatternScaling(
                        mname,
                        {"tas": 1, "pr": 1},
                        lambda key: training_data_dict[mname][key],
                        from_file=False,
                        exp_list=["base", "co2x4", "sulxanom"],
                        anom_timescales={"tas": 1, "pr": 1}
                    )

    for nv,var in enumerate(flds):
        tser_og_ghg_co2x4[nv,:] = global_mean(tmp.dacanom[var][tmp.exp_list.index('co2x4'), :100, :, :])
        tser_og_aer_sulxanom[nv,:] = global_mean(tmp.dacanom[var][tmp.exp_list.index('sulxanom'), :250, :, :])


    # create GHG modes and spatial pattern
    for ntsg,tsg in enumerate(timescales_ghg):
        for ntsa,tsa in enumerate(timescales_aer):

            CMIP_pattern_dict_tscales = MeteorPatternScaling(
                    mname,
                    {"tas": tsg, "pr": tsg},
                    lambda key: training_data_dict[mname][key],
                    from_file=False,
                    exp_list=["base", "co2x4", "sulxanom"],
                    anom_timescales={"tas": tsa, "pr": tsa}
                )

            # calculate goodness of the fit vs. orginal data
            for nv,var in enumerate(flds):
                # ghg
                tser_og_ghg[nv,nm,ntsg,:] = tser_og_ghg_co2x4[nv,:] #global_mean(CMIP_pattern_dict_tscales.dacanom[var][CMIP_pattern_dict_tscales.exp_list.index('co2x4'), :100, :, :])
                tser_fit_ghg[nv,nm,ntsg,:] = global_mean(recon(CMIP_pattern_dict_tscales.pattern_dict['co2x4'][var]['pattern_full']))
                mae_ghg[nv,nm,ntsg] = np.abs(tser_og_ghg[nv,nm,ntsg,:]-tser_fit_ghg[nv,nm,ntsg,:]).mean()
                rmse_ghg[nv,nm,ntsg] = np.sqrt(np.mean((tser_og_ghg[nv,nm,ntsg,:]-tser_fit_ghg[nv,nm,ntsg,:])**2))
                # ghg+aer
                tser_og_aer[nv,nm,ntsg,ntsa,:] = tser_og_aer_sulxanom[nv,:] #global_mean(CMIP_pattern_dict_tscales.dacanom[var][CMIP_pattern_dict_tscales.exp_list.index('sulxanom'), :250, :, :])
                tser_fit_aer[nv,nm,ntsg,ntsa,:] = global_mean(np.sum(CMIP_pattern_dict_tscales.predict_from_combined_experiment(em_data, conc_data, ["pr", "tas"], return_patterns_per_mode=True)[var][:,100:350,:,:],0))
                mae_aer[nv,nm,ntsg,ntsa] = np.abs(tser_og_aer[nv,nm,ntsg,ntsa,:]-tser_fit_aer[nv,nm,ntsg,ntsa,:]).mean()
                rmse_aer[nv,nm,ntsg,ntsa] = np.sqrt(np.mean((tser_og_aer[nv,nm,ntsg,ntsa,:]-tser_fit_aer[nv,nm,ntsg,ntsa,:])**2))
                


In [ ]:
%%capture

# prepare figure canvas
fig = plt.figure(figsize=(12,len(models)*2.5))
gs = GridSpec(len(models), 2, figure=fig)

for nm,mname in enumerate(models):

    sub = fig.add_subplot(gs[nm, 0])
    sub.set_xlabel("Years")
    sub.set_ylabel("$\Delta$GMST [K]")
    sub.plot(tser_og_aer[0,nm,0,0,:])
    sub.plot(tser_fit_aer[0,nm,0,0,:], label="ghg:1, aer:1")
    sub.plot(tser_fit_aer[0,nm,1,0,:], label="ghg:2, aer:1")
    sub.plot(tser_fit_aer[0,nm,2,0,:], label="ghg:3, aer:1")
    sub.plot(tser_fit_aer[0,nm,0,1,:], label="ghg:1, aer:2")
    sub.plot(tser_fit_aer[0,nm,1,1,:], label="ghg:2, aer:2")
    sub.plot(tser_fit_aer[0,nm,2,1,:], label="ghg:3, aer:2")
    sub.plot(tser_fit_aer[0,nm,0,2,:], label="ghg:1, aer:3")
    sub.plot(tser_fit_aer[0,nm,1,2,:], label="ghg:2, aer:3")
    sub.plot(tser_fit_aer[0,nm,2,2,:], label="ghg:3, aer:3")
    sub.legend(frameon=False, loc="upper left", ncol=2, prop={'size': 9})
    sub.annotate(mname, xy=(0.7, 0.04), xycoords='axes fraction', fontsize=10, ha='left', va='center', weight="bold")

    sub = fig.add_subplot(gs[nm, 1])
    sub.set_xlabel("Years")
    sub.set_ylabel("$\Delta$GMP [kg m-2 s-1]")
    sub.plot(tser_og_aer[1,nm,0,0,:])
    sub.plot(tser_fit_aer[1,nm,0,0,:], label="ghg:1, aer:1")
    sub.plot(tser_fit_aer[1,nm,1,0,:], label="ghg:2, aer:1")
    sub.plot(tser_fit_aer[1,nm,2,0,:], label="ghg:3, aer:1")
    sub.plot(tser_fit_aer[1,nm,0,1,:], label="ghg:1, aer:2")
    sub.plot(tser_fit_aer[1,nm,1,1,:], label="ghg:2, aer:2")
    sub.plot(tser_fit_aer[1,nm,2,1,:], label="ghg:3, aer:2")
    sub.plot(tser_fit_aer[1,nm,0,2,:], label="ghg:1, aer:3")
    sub.plot(tser_fit_aer[1,nm,1,2,:], label="ghg:2, aer:3")
    sub.plot(tser_fit_aer[1,nm,2,2,:], label="ghg:3, aer:3")
    sub.legend(frameon=False, loc="upper left", ncol=2, prop={'size': 9})

plt.savefig(os.path.join(figdir, "CMIP_timescales_fit_individual.jpg"), bbox_inches='tight')

In [ ]:
cbar_kws_tas = {"shrink": 0.9, "aspect": 15, "pad": 0.2, "label": "RMSE"}
cbar_kws_pr = {"shrink": 0.9, "aspect": 15, "pad": 0.2, "label": "RMSE [1e-7]"}

fig = plt.figure(figsize=(10,6))

nrows = 2
ncols = 8
X = [ (1,3), (4,4), (5,8),
      (9,11), (12,12), (13,16)]

## first row
sub = fig.add_subplot(2, ncols, X[0])
sub.set_ylabel('RMSE GMST [K]')
sub.set_xlabel('# \u03C4$_{GHG}$')
sub.set_ylim(0,0.6)
sub.set_xticks([0,1,2,3])
sub.set_xticklabels([1,2,3,4])
for m in np.arange(0,rmse_ghg.shape[1]):
    if m == 0: sub.plot(rmse_ghg[0,m,:], color='tab:red', label="Individual ESMs")
    sub.plot(rmse_ghg[0,m,:], color='tab:red',alpha=0.5,zorder=0)
    sub.scatter(np.arange(0,4), rmse_ghg[0,m,:], color='tab:red', s=15, edgecolor="black")
sub.plot(np.nanmean(rmse_ghg[0,:,:],0), color='black',lw=1.5)
sub.scatter(np.arange(0,4), np.nanmean(rmse_ghg[0,:,:],0), color='black', s=30, edgecolor="black", label="CMIP6 MMM")
sub.legend(frameon=False)
sub.annotate('a', xy=(-0.06, 1.13), xycoords='axes fraction', fontsize=12, ha='left', va='center', weight="bold")

sub = fig.add_subplot(2, ncols, X[1])
sns.heatmap(pd.DataFrame(np.nanmean(rmse_ghg[0,:,:],0)), ax=sub, annot=True, cmap='Reds', yticklabels=[1,2,3,4], xticklabels=[0], 
            vmin = 0.1, vmax = 0.4, linewidth=1, square=True, cbar=False)
sub.set_ylabel('# \u03C4$_{GHG}$')
sub.set_xlabel('# \u03C4$_{aer}$')
sub.yaxis.set_label_position("right")
sub.yaxis.tick_right()
sub.annotate('b', xy=(-0.26, 1.13), xycoords='axes fraction', fontsize=12, ha='left', va='center', weight="bold")

sub = fig.add_subplot(2, ncols, X[2])
sns.heatmap(np.nanmean(rmse_aer[0,:,:,:],0), annot=True, cmap='Reds', xticklabels=[0,1,2,3,4], yticklabels=[1,2,3,4],
            vmin = 0.1, vmax = 0.4, linewidth=1, square=True, cbar=True, cbar_kws=cbar_kws_tas)
sub.set_ylabel('# \u03C4$_{GHG}$')
sub.set_xlabel('# \u03C4$_{aer}$')
sub.yaxis.set_label_position("right")
sub.yaxis.tick_right()
sub.annotate('c', xy=(-0.06, 1.13), xycoords='axes fraction', fontsize=12, ha='left', va='center', weight="bold")

## second row
sub = fig.add_subplot(2, ncols, X[3])
sub.set_ylabel('RMSE GMP [kg m-2 s-1]')
sub.set_xlabel('# \u03C4$_{GHG}$')
sub.set_ylim(0,5e-7)
sub.set_xticks([0,1,2,3])
sub.set_xticklabels([1,2,3,4])
for m in np.arange(0,rmse_ghg.shape[1]):
    if m == 0: sub.plot(rmse_ghg[1,m,:], color='teal', label="Individual ESMs")
    sub.plot(rmse_ghg[1,m,:], color='teal',alpha=0.5,zorder=0)
    sub.scatter(np.arange(0,4), rmse_ghg[1,m,:], color='teal', s=15, edgecolor="black")
sub.plot(np.nanmean(rmse_ghg[1,:,:],0), color='black',lw=1.5)
sub.scatter(np.arange(0,4), np.nanmean(rmse_ghg[1,:,:],0), color='black', s=30, edgecolor="black", label="CMIP6 MMM")
sub.legend(frameon=False)
sub.annotate('d', xy=(-0.06, 1.13), xycoords='axes fraction', fontsize=12, ha='left', va='center', weight="bold")

sub = fig.add_subplot(2, ncols, X[4])
sns.heatmap(pd.DataFrame(np.nanmean(rmse_ghg[1,:,:],0))*1e7, ax=sub, annot=True, cmap='GnBu', yticklabels=[1,2,3,4], xticklabels=[0], 
            vmin = 1, vmax = 3, linewidth=1, square=True, cbar=False)
sub.set_ylabel('# \u03C4$_{GHG}$')
sub.set_xlabel('# \u03C4$_{aer}$')
sub.yaxis.set_label_position("right")
sub.yaxis.tick_right()
sub.annotate('e', xy=(-0.26, 1.13), xycoords='axes fraction', fontsize=12, ha='left', va='center', weight="bold")

sub = fig.add_subplot(2, ncols, X[5])

sns.heatmap(np.nanmean(rmse_aer[1,:,:,:],0)*1e7, annot=True, cmap='GnBu', xticklabels=[0,1,2,3,4], yticklabels=[1,2,3,4],
            vmin = 1, vmax = 3, linewidth=1, square=True, cbar=True, cbar_kws=cbar_kws_pr)
sub.set_ylabel('# \u03C4$_{GHG}$')
sub.set_xlabel('# \u03C4$_{aer}$')
sub.yaxis.set_label_position("right")
sub.yaxis.tick_right()
sub.annotate('f', xy=(-0.06, 1.13), xycoords='axes fraction', fontsize=12, ha='left', va='center', weight="bold")

fig.subplots_adjust(wspace=0.6, hspace=0.6)

plt.savefig(os.path.join(figdir, "fit_timescale_evaluation.pdf"), bbox_inches='tight')

In [ ]:
labels = (["a","b","c","d","e"],
          ["f","g","h","i","j"],
          ["k","l","m","n","o"],
          ["p","q","r","s","t"],)

fig = plt.figure(figsize=(12,10))
gs = GridSpec(len(timescales_ghg), len(timescales_aer), figure=fig)

nv = 0
for ntscg,tscg in enumerate(timescales_ghg):
    for ntsca,tsca in enumerate(timescales_aer):
        sub = fig.add_subplot(gs[ntscg, ntsca])
        sub.set_title("T$_{GHG}$="+str(tscg)+"; T$_{aer}$="+str(tsca), size=8)
        sub.set_xlabel("Original [K]")
        sub.set_ylabel("Reconstruction [K]")
        sub.set_ylim(-1,5)
        sub.set_xlim(-1,5)
        #sub.set_xticks([0,1,2,3,4])
        #sub.set_xticklabels([1,2,3,4,5])
        for m in np.arange(0,tser_og_aer.shape[1]):
            sub.scatter(tser_og_aer[nv,m,ntscg,ntsca,:], tser_fit_aer[nv,m,ntscg,ntsca,:], color="gray", alpha=0.5, edgecolor="white")
        sub.plot([-100,100],[-100,100], color="black", lw=0.7, linestyle="--")
        sub.annotate(labels[ntscg][ntsca], xy=(-0.09, 1.17), xycoords='axes fraction', fontsize=13, ha='left', va='center', weight="bold")
        sub.annotate("RMSE="+"{:.2f}".format(np.nanmean(rmse_aer[nv,:,ntscg, ntsca])), xy=(0.04, 0.9), xycoords='axes fraction', fontsize=8, ha='left', va='center')
        sub.annotate("MAE="+"{:.2f}".format(np.nanmean(mae_aer[nv,:,ntscg, ntsca])), xy=(0.04, 0.8), xycoords='axes fraction', fontsize=8, ha='left', va='center')
        
fig.subplots_adjust(wspace=0.45, hspace=0.65)
plt.savefig(os.path.join(figdir, "fit_timescale_evaluation_scatter_tas.pdf"), bbox_inches='tight')

In [ ]:
labels = (["a","b","c","d","e"],
          ["f","g","h","i","j"],
          ["k","l","m","n","o"],
          ["p","q","r","s","t"],)

fig = plt.figure(figsize=(12,10))
gs = GridSpec(len(timescales_ghg), len(timescales_aer), figure=fig)

nv = 1
for ntscg,tscg in enumerate(timescales_ghg):
    for ntsca,tsca in enumerate(timescales_aer):
        sub = fig.add_subplot(gs[ntscg, ntsca])
        sub.set_title("T$_{GHG}$="+str(tscg)+"; T$_{aer}$="+str(tsca), size=8)
        sub.set_xlabel("Original")
        sub.set_ylabel("Reconstruction")
        sub.set_ylim(-2e-6,3.5e-6)
        sub.set_xlim(-2e-6,3.5e-6)
        #sub.set_xticks([0,1,2,3,4])
        #sub.set_xticklabels([1,2,3,4,5])
        for m in np.arange(0,tser_og_aer.shape[1]):
            sub.scatter(tser_og_aer[nv,m,ntscg,ntsca,:], tser_fit_aer[nv,m,ntscg,ntsca,:], color="gray", alpha=0.5, edgecolor="white")
        sub.plot([-100,100],[-100,100], color="black", lw=0.7, linestyle="--")
        sub.annotate(labels[ntscg][ntsca], xy=(-0.09, 1.17), xycoords='axes fraction', fontsize=13, ha='left', va='center', weight="bold")
        sub.annotate("RMSE="+"{:.2f}".format(np.nanmean(rmse_aer[nv,:,ntscg, ntsca])*1e6), xy=(0.04, 0.9), xycoords='axes fraction', fontsize=8, ha='left', va='center')
        sub.annotate("MAE="+"{:.2f}".format(np.nanmean(mae_aer[nv,:,ntscg, ntsca])*1e6), xy=(0.04, 0.8), xycoords='axes fraction', fontsize=8, ha='left', va='center')
        
fig.subplots_adjust(wspace=0.45, hspace=0.65)
plt.savefig(os.path.join(figdir, "fit_timescale_evaluation_scatter_pr.pdf"), bbox_inches='tight')